In [ ]:
def train_model(train_df,test_df):
    import pandas as pd
    import numpy as np
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.metrics import accuracy_score, classification_report, confusion_metrics
    from sklearn.preprocessing import LabelEncoder
    from sklearn.impute import SimpleImputer

    df = train_df.copy()
    test = test_df.copy()

    y = df['target']
    X = df.drop('target',axis=1)

    drop_cols = [col for col in X.columns if "id" in col.lower()]
    
    X = X.drop(drop_cols, axis=1, errors='ignore')

    num_cols =  X.select_dtypes(include = np.number).columns
    cat_cols = X.select_dtypes(include = 'object').columns

    num_imputer = SimpleImputer(strategy='mean')
    X[num_cols] = num_imputer.fit_transform(X[num_cols])
    test[num_cols] = num_imputer.transform(test[num_cols])

    cat_imputer = SimpleImputer(strategy='most_frequent')
    X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])
    test[cat_cols] = cat_imputer.transform(test[cat_cols])

    for col in cat_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        test[col] = le.transform(test[col])
    
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

    models = [
        LogisticRegression(max_iter=1000, random_state=42),
        RandomForestClassifier(n_estimators=100, random_state=42),
        DecisionTreeClassifier(random_state=42)
    ]

    best_model = None
    best_score = 0 
    for m in models:
        m.fit(X_train,y_train)
        pred = m.predict(X_test)
        accscore = accuracy_score(y_test,pred)
        print(f"{m.__class__.__name__} Accuracy: {accscore:.4f}")
        if accscore > best_score:
            best_score = accscore
            best_model = m  

    print(f"Best Model: {best_model.__class__.__name__}")
    
    best_model.fit(X,y)
    test_pred = best_model.predict(test)
    return test_pred